# CosyVoice 3 전용 · 목소리 20종 · v1.0.1

**CosyVoice 2와 별도의 GPU 런타임에서 실행하세요.**

- 남성 10명 + 여성 10명의 서로 다른 공개 참고 음성. 업로드·Gemini API 키 없이 선택합니다.
- 기본 20종은 VCTK 영어 녹음의 음색으로 한국어를 합성합니다. 한국어 억양과 음질은 직접 확인해야 합니다.
- 원하는 한국어 목소리는 사이트의 **내 한국어 녹음**에서 등록할 수 있습니다.
- 모델: Fun-CosyVoice3-0.5B-2512, **FP32**, 공식 샘플링·생성 단계 사용.
- 런타임 → 런타임 유형 변경 → **T4 GPU**를 선택하세요.
- **1 → 2 → 4**: 공유 사이트 연결. **1 → 2 → 3**: 코랩 안에서 직접 생성.
- 1·2번은 설치·모델 준비이며 미리듣기나 테스트 음성을 만들지 않습니다.
- 새로 추가한 버전 3 코랩입니다. 버전 2 주소 칸에는 이 주소를 넣지 마세요.

무료 Colab의 자원·유휴·외부 웹 사용 제한은 적용됩니다. 외부 연결이 제한되면 3번으로 직접 생성하세요. [공식 안내](https://research.google.com/colaboratory/faq.html)

**v1.0.1 화면 출력 수정:** 설치·서버 준비 로그와 사이트 연결 주소를 코랩 화면에 직접 표시합니다. 4번에서 ‘서버 정상·외부 연결 정상’이 반복되는 것은 유지 중이라는 뜻입니다. 주소를 사이트에 넣고 사용하세요.


In [ ]:
#@title 1. CosyVoice 3 전용 설치 (첫 실행만 다운로드)
from pathlib import Path
import subprocess, sys
ROOT = Path('/content/voice_studio_cosy3_v1')
ROOT.mkdir(parents=True, exist_ok=True)
(ROOT / 'cosy3_colab_server.py').write_text('"""Independent CosyVoice 3 Colab service; never starts or changes CosyVoice 2."""\nimport argparse\nimport base64\nimport hashlib\nimport json\nimport logging\nimport math\nimport os\nfrom pathlib import Path\nimport queue\nimport re\nimport secrets\nimport shutil\nimport socket\nimport subprocess\nimport sys\nimport threading\nimport time\nimport traceback\nimport urllib.request\n\nfrom cosy3_voicebank_catalog import (MODEL_ID, MODEL_REVISION, SOURCE_REVISION,\n    SERVICE, PROTOCOL, BANK_REVISION, VOICEBANK, ATTRIBUTION)\n\nROOT = Path(\'/content/voice_studio_cosy3_v1\')\nSOURCE = ROOT / \'CosyVoice\'\nPYTHON = ROOT / \'venv/bin/python\'\nMODEL = SOURCE / \'pretrained_models/Fun-CosyVoice3-0.5B\'\nSTATE = ROOT / \'state.json\'\nVERSION = \'1.0.0\'\nMAX_REQUEST_BYTES = 128 * 1024\nMAX_REFERENCE_BYTES = 10 * 1024 * 1024\n\n\ndef save_json(path, data):\n    path = Path(path)\n    temporary = path.with_suffix(\'.tmp\')\n    temporary.write_text(json.dumps(data, ensure_ascii=False), encoding=\'utf-8\')\n    temporary.chmod(0o600)\n    os.replace(temporary, path)\n\n\ndef read_state():\n    try:\n        return json.loads(STATE.read_text())\n    except (OSError, ValueError):\n        return {}\n\n\ndef run(args, label):\n    print(\'\\n▶ \' + label, flush=True)\n    with (ROOT / \'setup.log\').open(\'a\', encoding=\'utf-8\') as log:\n        log.write(\'\\n▶ \' + label + \'\\n\')\n        process = subprocess.Popen([str(arg) for arg in args], stdout=subprocess.PIPE,\n            stderr=subprocess.STDOUT, text=True, bufsize=1)\n        try:\n            for line in process.stdout:\n                print(line, end=\'\', flush=True)\n                log.write(line)\n                log.flush()\n            code = process.wait()\n        except BaseException:\n            process.terminate()\n            raise\n    if code:\n        raise RuntimeError(f\'{label} 실패 (종료 코드 {code}). 위 오류를 확인해주세요.\')\n\n\ndef local_request(path, payload=None, timeout=15):\n    state = read_state()\n    if not state.get(\'base\'):\n        raise RuntimeError(\'먼저 2번 셀에서 CosyVoice 3 서버를 준비해주세요.\')\n    req = urllib.request.Request(state[\'base\'] + path,\n        data=json.dumps(payload).encode() if payload is not None else None,\n        headers={\'Content-Type\': \'application/json\'})\n    with urllib.request.urlopen(req, timeout=timeout) as response:\n        return json.load(response)\n\n\ndef process_alive(pid):\n    if not isinstance(pid, int):\n        return False\n    try:\n        return not Path(f\'/proc/{pid}/stat\').read_text().split(\') \', 1)[1].startswith(\'Z\')\n    except (OSError, IndexError):\n        return False\n\n\ndef setup():\n    if not Path(\'/content\').is_dir():\n        raise RuntimeError(\'이 파일은 Google Colab에서 실행해주세요.\')\n    ROOT.mkdir(parents=True, exist_ok=True)\n    marker = ROOT / \'installed.json\'\n    if marker.is_file() and PYTHON.is_file():\n        print(\'✅ CosyVoice 3 설치 파일을 재사용합니다. 2번 서버 준비를 실행하세요.\', flush=True)\n        return\n    if shutil.which(\'nvidia-smi\') is None:\n        raise RuntimeError(\'런타임 → 런타임 유형 변경 → T4 GPU를 먼저 선택해주세요.\')\n    run([\'apt-get\', \'update\', \'-qq\'], \'오디오 도구 준비\')\n    run([\'apt-get\', \'install\', \'-y\', \'-qq\', \'ffmpeg\', \'sox\', \'libsox-dev\',\n         \'libsndfile1\', \'build-essential\', \'git\'], \'오디오 도구 설치\')\n    if not PYTHON.is_file():\n        run([sys.executable, \'-m\', \'pip\', \'install\', \'--disable-pip-version-check\', \'uv\'], \'독립 Python 준비\')\n        run([sys.executable, \'-m\', \'uv\', \'python\', \'install\', \'3.10\'], \'Python 3.10 설치\')\n        run([sys.executable, \'-m\', \'uv\', \'venv\', \'--python\', \'3.10\', \'--seed\', ROOT / \'venv\'], \'버전 3 전용 실행 환경\')\n    if not (SOURCE / \'.git\').is_dir():\n        run([\'git\', \'clone\', \'--filter=blob:none\', \'https://github.com/QwenAudio/CosyVoice.git\', SOURCE], \'공식 소스 받기\')\n    run([\'git\', \'-C\', SOURCE, \'checkout\', \'--detach\', SOURCE_REVISION], \'소스 버전 고정\')\n    run([\'git\', \'-C\', SOURCE, \'submodule\', \'update\', \'--init\', \'--recursive\'], \'Matcha-TTS 준비\')\n    requirements = []\n    for line in (SOURCE / \'requirements.txt\').read_text().splitlines():\n        if line.startswith((\'deepspeed\', \'tensorrt\', \'gradio\', \'fastapi-cli\')):\n            continue\n        if line.startswith(\'diffusers==\'):\n            line = \'diffusers==0.32.2\'\n        requirements.append(line)\n    requirements += [\'huggingface-hub==0.30.2\', \'setuptools<81\']\n    req = ROOT / \'inference-requirements.txt\'\n    req.write_text(\'\\n\'.join(requirements) + \'\\n\')\n    build = ROOT / \'build-constraints.txt\'\n    build.write_text(\'setuptools<81\\n\')\n    run([PYTHON, \'-m\', \'pip\', \'install\', \'pip>=25.3,<26\', \'setuptools<81\', \'wheel\', \'Cython<4\'], \'설치 도구 준비\')\n    run([PYTHON, \'-m\', \'pip\', \'install\', \'torch==2.3.1\', \'torchaudio==2.3.1\',\n         \'--index-url\', \'https://download.pytorch.org/whl/cu121\'], \'GPU 오디오 라이브러리 설치\')\n    run([PYTHON, \'-m\', \'pip\', \'install\', \'-r\', req, \'--build-constraint\', build], \'CosyVoice 3 라이브러리 설치\')\n    download = (\'from huggingface_hub import snapshot_download; \'\n        f\'snapshot_download({MODEL_ID!r}, revision={MODEL_REVISION!r}, local_dir={str(MODEL)!r}, \'\n        \'allow_patterns=["cosyvoice3.yaml", "llm.pt", "flow.pt", "hift.pt", "campplus.onnx", \'\n        \'"speech_tokenizer_v3.onnx", "CosyVoice-BlankEN/*"])\')\n    run([PYTHON, \'-c\', download], \'CosyVoice 3 모델 다운로드 · 첫 설치 때만 필요\')\n    (ROOT / \'VOICE_ATTRIBUTION.txt\').write_text(ATTRIBUTION, encoding=\'utf-8\')\n    save_json(marker, {\'version\': VERSION, \'model\': MODEL_ID, \'revision\': MODEL_REVISION})\n    print(\'✅ 1번 설치 완료. 2번 서버 준비를 실행하세요.\', flush=True)\n\n\ndef start():\n    if not (ROOT / \'installed.json\').is_file() or not PYTHON.is_file():\n        raise RuntimeError(\'먼저 1번 설치를 끝까지 실행해주세요.\')\n    state = read_state()\n    if process_alive(state.get(\'pid\')):\n        print(\'▶ 이미 실행한 CosyVoice 3 서버 준비를 이어서 확인합니다.\', flush=True)\n        wait_ready(state[\'pid\'])\n        return\n    # Distinct folders do not make two GPU models independent on the same VM.\n    for other in (\'/content/ai_voice_dual_v1/cosyvoice\', \'/content/voice_studio_qwen_bank_v1\',\n                  \'/content/ai_voice_sovits_v4_1\'):\n        path = Path(other) / \'state.json\'\n        if path.is_file():\n            try:\n                if process_alive(json.loads(path.read_text()).get(\'pid\')):\n                    raise RuntimeError(\'이 런타임에 다른 음성 엔진이 실행 중입니다. CosyVoice 3는 별도의 GPU 런타임에서 열어주세요.\')\n            except (ValueError, OSError):\n                pass\n    with socket.socket() as sock:\n        sock.bind((\'127.0.0.1\', 0))\n        port = sock.getsockname()[1]\n    token = secrets.token_urlsafe(32)\n    state = {\'port\': port, \'token\': token, \'base\': f\'http://127.0.0.1:{port}/v1/{token}\',\n             \'service\': SERVICE, \'version\': VERSION}\n    env = dict(os.environ, PYTHONUNBUFFERED=\'1\', MPLBACKEND=\'Agg\', TOKENIZERS_PARALLELISM=\'false\',\n               OMP_NUM_THREADS=\'1\', MKL_NUM_THREADS=\'1\', OPENBLAS_NUM_THREADS=\'1\',\n               NUMEXPR_NUM_THREADS=\'1\', OMP_WAIT_POLICY=\'PASSIVE\')\n    env[\'PYTHONPATH\'] = os.pathsep.join([str(ROOT), str(SOURCE), str(SOURCE / \'third_party/Matcha-TTS\')])\n    libs = [str(p) for p in (ROOT / \'venv/lib/python3.10/site-packages/nvidia\').glob(\'*/lib\')]\n    env[\'LD_LIBRARY_PATH\'] = os.pathsep.join(libs + [env.get(\'LD_LIBRARY_PATH\', \'\')])\n    save_json(STATE, state)\n    with (ROOT / \'server.log\').open(\'w\') as log:\n        proc = subprocess.Popen([str(PYTHON), \'-u\', str(Path(__file__).resolve()), \'serve\'],\n            cwd=SOURCE, env=env, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)\n    state[\'pid\'] = proc.pid\n    save_json(STATE, state)\n    wait_ready(proc.pid)\n\n\ndef wait_ready(pid):\n    began, last = time.monotonic(), \'\'\n    print(\'▶ 모델 준비 중입니다. 음성은 생성하지 않습니다.\', flush=True)\n    while time.monotonic() - began < 1800:\n        try:\n            info = local_request(\'/health\', timeout=5)\n            if info.get(\'ready\') and info.get(\'service\') == SERVICE and info.get(\'protocol\') == PROTOCOL:\n                print(f"✅ CosyVoice 3 v{VERSION} 준비 완료 · {info[\'gpu\']} · FP32 · 목소리 20종", flush=True)\n                print(\'사이트를 연결하려면 4번을, 코랩에서 직접 만들려면 3번을 실행하세요.\', flush=True)\n                return\n        except (OSError, ValueError):\n            pass\n        log = (ROOT / \'server.log\').read_text(errors=\'replace\')[-2400:]\n        if log != last:\n            print(log if not last else log[-1000:], flush=True)\n            last = log\n        if not process_alive(pid):\n            raise RuntimeError(\'CosyVoice 3 서버가 중단되었습니다. 위 실제 오류 로그를 확인해주세요.\')\n        time.sleep(3)\n    raise RuntimeError(\'서버 준비가 30분을 넘었습니다. 5번 로그에서 마지막 단계를 확인해주세요.\')\n\n\ndef tunnel():\n    info = local_request(\'/health\')\n    if not info.get(\'ready\') or info.get(\'service\') != SERVICE:\n        raise RuntimeError(\'먼저 2번 서버 준비를 끝내주세요.\')\n    state = read_state()\n    if process_alive(state.get(\'tunnel_pid\')) and state.get(\'public_url\'):\n        print(\'✅ CosyVoice 3 프로그램 연결 주소:\\n\' + state[\'public_url\'], flush=True)\n        return\n    binary = ROOT / \'cloudflared\'\n    if not binary.is_file():\n        urllib.request.urlretrieve(\'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64\', binary)\n        binary.chmod(0o700)\n    log_path = ROOT / \'tunnel.log\'\n    with log_path.open(\'w\') as log:\n        process = subprocess.Popen([str(binary), \'tunnel\', \'--url\', f"http://127.0.0.1:{state[\'port\']}",\n            \'--no-autoupdate\'], stdout=log, stderr=subprocess.STDOUT, start_new_session=True)\n    state[\'tunnel_pid\'] = process.pid\n    save_json(STATE, state)\n    for _ in range(60):\n        match = re.search(r\'https://[a-z0-9-]+\\.trycloudflare\\.com\', log_path.read_text(errors=\'replace\'))\n        if match:\n            state[\'public_url\'] = match.group() + \'/v1/\' + state[\'token\']\n            save_json(STATE, state)\n            print(\'✅ CosyVoice 3 프로그램 연결 주소:\\n\' + state[\'public_url\'], flush=True)\n            print(\'사이트의 CosyVoice 3 주소 칸에 넣으세요. CosyVoice 2 주소와 별개입니다.\', flush=True)\n            return\n        if process.poll() is not None:\n            break\n        time.sleep(1)\n    raise RuntimeError(\'외부 연결 주소를 만들지 못했습니다. 5번 로그를 확인해주세요. 모델은 준비되어 있습니다.\')\n\n\ndef serve():\n    import soundfile as sf\n    from fastapi import FastAPI, HTTPException, Request\n    from fastapi.responses import FileResponse\n    import uvicorn\n    from cosy3_model import Cosy3Model, spoken\n    state = read_state()\n    model = Cosy3Model(MODEL, ROOT)\n    app = FastAPI(docs_url=None, redoc_url=None, openapi_url=None)\n    jobs, lock, pending = {}, threading.RLock(), queue.Queue(maxsize=32)\n    job_dir, ref_dir = ROOT / \'jobs\', ROOT / \'references\'\n    job_dir.mkdir(exist_ok=True)\n    ref_dir.mkdir(exist_ok=True)\n\n    def authorize(token):\n        if not secrets.compare_digest(token, state[\'token\']):\n            raise HTTPException(404, \'주소를 확인해주세요.\')\n\n    def public(job):\n        return {key: value for key, value in job.items() if key not in (\'payload\', \'identity\', \'file\')}\n\n    def persist(job_id):\n        save_json(job_dir / (job_id + \'.json\'), jobs[job_id])\n\n    def worker():\n        while True:\n            job_id = pending.get()\n            try:\n                with lock:\n                    job = jobs[job_id]\n                    if job[\'status\'] == \'cancelled\':\n                        continue\n                    job.update(status=\'generating\', phase=\'음성 계산\', started=time.time())\n                    persist(job_id)\n                    payload = dict(job[\'payload\'])\n                def progress(done, total, cached):\n                    with lock:\n                        job.update(chunks_done=done, chunks=total, reference_cached=cached)\n                started = time.monotonic()\n                reference = ref_dir / (payload.pop(\'reference_id\') + \'.wav\') if payload.get(\'reference_id\') else None\n                payload.pop(\'reference_id\', None)\n                audio, sample_rate = model.generate(**payload, ref_path=str(reference) if reference else \'\', progress=progress)\n                audio_path = job_dir / (job_id + \'.wav\')\n                temp = audio_path.with_suffix(\'.part.wav\')\n                sf.write(temp, audio, sample_rate, subtype=\'PCM_16\')\n                os.replace(temp, audio_path)\n                with lock:\n                    job.update(status=\'done\', phase=\'완료\', file=str(audio_path),\n                        synthesis_seconds=time.monotonic() - started, audio_seconds=len(audio) / sample_rate,\n                        sample_rate=sample_rate, finished=time.time())\n                    persist(job_id)\n            except Exception as exc:\n                traceback.print_exc()\n                with lock:\n                    jobs[job_id].update(status=\'error\', error_code=\'memory\' if \'out of memory\' in str(exc).lower() else \'generation\',\n                        phase=\'오류\', message=str(exc)[:500], finished=time.time())\n                    persist(job_id)\n            finally:\n                pending.task_done()\n\n    # One model compute lane; queued next line starts before HTTP download/save.\n    # Avoid claiming unmeasured GPU parallelism or changing inference precision.\n    threading.Thread(target=worker, name=\'cosy3-inference\', daemon=True).start()\n\n    @app.get(\'/v1/{token}/health\')\n    def health(token: str):\n        authorize(token)\n        return dict(service=SERVICE, protocol=PROTOCOL, version=VERSION, model=MODEL_ID,\n            model_revision=MODEL_REVISION, bank_revision=BANK_REVISION, ready=True,\n            gpu=model.gpu, precision=\'FP32\', sample_rate=model.sample_rate,\n            voice_count=len(VOICEBANK), voices=VOICEBANK, gpu_concurrency=1,\n            queued=pending.qsize(), max_requests=4)\n\n    async def body(request, maximum):\n        data = bytearray()\n        async for chunk in request.stream():\n            data.extend(chunk)\n            if len(data) > maximum:\n                raise HTTPException(413, \'요청 파일이 너무 큽니다.\')\n        return bytes(data)\n\n    @app.post(\'/v1/{token}/references/{reference_id}\')\n    async def upload_reference(token: str, reference_id: str, request: Request):\n        authorize(token)\n        if not re.fullmatch(r\'[a-f0-9]{64}\', reference_id):\n            raise HTTPException(400, \'참고 음성 ID 오류\')\n        data = await body(request, MAX_REFERENCE_BYTES)\n        if hashlib.sha256(data).hexdigest() != reference_id:\n            raise HTTPException(400, \'참고 음성 파일이 완전하지 않습니다.\')\n        path = ref_dir / (reference_id + \'.wav\')\n        if path.is_file():\n            return dict(reference_id=reference_id)\n        import io\n        import numpy as np\n        try:\n            audio, rate = sf.read(io.BytesIO(data), dtype=\'float32\', always_2d=True)\n            if not 3 <= len(audio) / rate <= 30 or not np.isfinite(audio).all() or np.max(np.abs(audio)) < 0.001:\n                raise ValueError(\'3~30초의 또렷한 녹음이 필요합니다.\')\n        except Exception:\n            raise HTTPException(400, \'참고 음성을 읽지 못했습니다. 3~30초 WAV 또는 FLAC 파일을 사용해주세요.\') from None\n        # Normalize the container only; preserve reference sample rate and values.\n        temporary = path.with_name(\'.\' + reference_id + \'.\' + secrets.token_hex(4) + \'.wav\')\n        sf.write(temporary, audio, rate, subtype=\'FLOAT\')\n        os.replace(temporary, path)\n        return dict(reference_id=reference_id)\n\n    def get_job(job_id):\n        if not re.fullmatch(r\'[a-f0-9]{32}\', job_id):\n            raise HTTPException(400, \'작업 ID 오류\')\n        if job_id not in jobs:\n            path = job_dir / (job_id + \'.json\')\n            if path.is_file():\n                saved = json.loads(path.read_text())\n                if saved.get(\'status\') in (\'queued\', \'generating\'):\n                    saved.update(status=\'error\', error_code=\'interrupted\', message=\'서버 재시작으로 중단된 작업입니다.\')\n                jobs[job_id] = saved\n        return jobs.get(job_id)\n\n    @app.post(\'/v1/{token}/jobs/{job_id}\')\n    async def submit(token: str, job_id: str, request: Request):\n        authorize(token)\n        if not re.fullmatch(r\'[a-f0-9]{32}\', job_id):\n            raise HTTPException(400, \'작업 ID 오류\')\n        try:\n            raw = json.loads(await body(request, MAX_REQUEST_BYTES))\n            if not isinstance(raw, dict):\n                raise ValueError(\'요청 형식 오류\')\n            voice = str(raw.get(\'voice\', \'\'))\n            if voice not in VOICEBANK and voice != \'custom\':\n                raise ValueError(\'목소리를 다시 선택해주세요.\')\n            speed = float(raw.get(\'speed\', 1))\n            if not math.isfinite(speed) or not 0.8 <= speed <= 1.2:\n                raise ValueError(\'속도 범위 오류\')\n            style = str(raw.get(\'style\', \'\'))\n            if len(style) > 1200 or \'<|\' in style or \'|>\' in style:\n                raise ValueError(\'스타일 지시 오류\')\n            payload = dict(text=spoken(raw.get(\'text\', \'\')), voice=voice, speed=speed, style=style,\n                           prompt_text=\'\', reference_id=\'\')\n            if voice == \'custom\':\n                ref_id = str(raw.get(\'reference_id\', \'\'))\n                if not re.fullmatch(r\'[a-f0-9]{64}\', ref_id) or not (ref_dir / (ref_id + \'.wav\')).is_file():\n                    raise ValueError(\'참고 음성을 먼저 등록해주세요.\')\n                payload.update(reference_id=ref_id, prompt_text=spoken(raw.get(\'prompt_text\', \'\'), 2000))\n        except (ValueError, TypeError) as exc:\n            raise HTTPException(400, str(exc)) from None\n        identity = hashlib.sha256(json.dumps(payload, sort_keys=True, ensure_ascii=False).encode()).hexdigest()\n        with lock:\n            old = get_job(job_id)\n            if old:\n                if old.get(\'identity\') != identity:\n                    raise HTTPException(409, \'같은 작업 ID에 다른 대사가 전달됐습니다.\')\n                return public(old)\n            if pending.full() or shutil.disk_usage(ROOT).free < 512 * 1024 * 1024:\n                raise HTTPException(429, \'대기 작업 또는 저장 공간이 부족합니다. 완료 작업을 저장한 뒤 이어서 생성해주세요.\')\n            job = dict(status=\'queued\', phase=\'대기\', payload=payload, identity=identity,\n                       created=time.time(), chunks_done=0, chunks=1, gpu_concurrency=1)\n            jobs[job_id] = job\n            persist(job_id)\n            pending.put_nowait(job_id)\n            return public(job)\n\n    @app.get(\'/v1/{token}/jobs/{job_id}\')\n    def status(token: str, job_id: str):\n        authorize(token)\n        with lock:\n            job = get_job(job_id)\n            if not job:\n                raise HTTPException(404, \'작업을 찾을 수 없습니다.\')\n            return public(job)\n\n    @app.post(\'/v1/{token}/jobs/{job_id}/cancel\')\n    def cancel(token: str, job_id: str):\n        authorize(token)\n        with lock:\n            job = get_job(job_id)\n            if not job:\n                raise HTTPException(404, \'작업을 찾을 수 없습니다.\')\n            if job[\'status\'] == \'queued\':\n                job.update(status=\'cancelled\', phase=\'취소\', finished=time.time())\n                persist(job_id)\n            return public(job)  # active calculation finishes and is saved.\n\n    @app.get(\'/v1/{token}/jobs/{job_id}/audio\')\n    def audio(token: str, job_id: str):\n        authorize(token)\n        with lock:\n            job = get_job(job_id)\n            if not job or job.get(\'status\') != \'done\' or not Path(job.get(\'file\', \'\')).is_file():\n                raise HTTPException(404, \'완료된 음성 파일이 없습니다.\')\n            return FileResponse(job[\'file\'], media_type=\'audio/wav\', filename=job_id + \'.wav\')\n\n    uvicorn.run(app, host=\'127.0.0.1\', port=state[\'port\'], access_log=False)\n\n\ndef direct(plan_path):\n    """Native Colab execution via the same private local API, no public tunnel."""\n    import io\n    import wave\n    import zipfile\n    plan = json.loads(Path(plan_path).read_text(encoding=\'utf-8\'))\n    if plan.get(\'format\') != \'voice-studio-cosy3-plan-v1\' or plan.get(\'bank_revision\') != BANK_REVISION:\n        raise ValueError(\'사이트에서 받은 CosyVoice 3 대본 ZIP을 사용해주세요.\')\n    info = local_request(\'/health\')\n    if info.get(\'service\') != SERVICE:\n        raise RuntimeError(\'CosyVoice 3 서버를 먼저 준비해주세요.\')\n    items = plan.get(\'items\', [])\n    if not items or len(items) > 4096:\n        raise ValueError(\'대본은 1~4096개 대사까지 지원합니다.\')\n    indices = [item[\'index\'] for item in items]\n    if len(set(indices)) != len(indices) or any(type(i) is not int or i < 1 for i in indices):\n        raise ValueError(\'대사 순번이 올바르지 않습니다.\')\n    references = plan.get(\'references\', {})\n    base = read_state()[\'base\']\n    for ref_id, encoded in references.items():\n        data = base64.b64decode(encoded, validate=True)\n        if len(data) > MAX_REFERENCE_BYTES or hashlib.sha256(data).hexdigest() != ref_id:\n            raise ValueError(\'대본의 참고 음성이 올바르지 않습니다.\')\n        req = urllib.request.Request(base + \'/references/\' + ref_id, data=data, method=\'POST\')\n        with urllib.request.urlopen(req, timeout=90) as response:\n            response.read()\n    output = ROOT / \'direct_results\'\n    output.mkdir(exist_ok=True)\n    attempts_path = output / \'job_ids.json\'\n    try:\n        attempts = json.loads(attempts_path.read_text())\n    except (OSError, ValueError):\n        attempts = {}\n    ordered = sorted(items, key=lambda item: item[\'index\'])\n    paths = []\n    for position, item in enumerate(ordered, 1):\n        payload = {key: value for key, value in item.items() if key not in (\'index\', \'speaker\')}\n        identity = json.dumps(payload, sort_keys=True, ensure_ascii=False)\n        key = hashlib.sha256((BANK_REVISION + identity).encode()).hexdigest()[:32]\n        job_id = attempts.get(key, key)\n        status = local_request(\'/jobs/\' + job_id, payload)\n        if status.get(\'status\') in (\'error\', \'cancelled\'):\n            # A fresh run explicitly resumes a previous failed line. Never\n            # automatically retry an error produced during this run.\n            job_id = secrets.token_hex(16)\n            attempts[key] = job_id\n            save_json(attempts_path, attempts)\n            status = local_request(\'/jobs/\' + job_id, payload)\n        started = time.monotonic()\n        while status.get(\'status\') != \'done\':\n            if status.get(\'status\') in (\'error\', \'cancelled\'):\n                raise RuntimeError(f"{item[\'index\']}번 중단: {status.get(\'message\', status[\'status\'])}")\n            if time.monotonic() - started > 1800:\n                raise RuntimeError(\'처리 대기가 30분을 넘었습니다. 5번 로그를 확인해주세요.\')\n            time.sleep(1)\n            status = local_request(\'/jobs/\' + job_id)\n        path = output / (f"{item[\'index\']:04d}_" + job_id + \'.wav\')\n        with urllib.request.urlopen(base + \'/jobs/\' + job_id + \'/audio\', timeout=90) as response:\n            path.write_bytes(response.read(128 * 1024 * 1024))\n        paths.append(path)\n        print(f"✅ {position}/{len(ordered)} · {item[\'speaker\']}", flush=True)\n    full = output / \'full_audio.wav\'\n    pause_ms = min(3000, max(0, int(plan.get(\'pause_ms\', 500))))\n    with wave.open(str(full), \'wb\') as dst:\n        for number, path in enumerate(paths):\n            with wave.open(str(path), \'rb\') as src:\n                if number == 0:\n                    params = src.getparams()\n                    dst.setparams(params)\n                elif src.getparams()[:3] != params[:3]:\n                    raise RuntimeError(\'음성 형식이 달라 합치기를 중단했습니다.\')\n                if number:\n                    dst.writeframes(b\'\\0\' * (params.nchannels * params.sampwidth * params.framerate * pause_ms // 1000))\n                dst.writeframes(src.readframes(src.getnframes()))\n    mp3 = output / \'full_audio.mp3\'\n    run([\'ffmpeg\', \'-hide_banner\', \'-loglevel\', \'error\', \'-y\', \'-i\', full, \'-b:a\', \'192k\', mp3], \'대본 순서대로 MP3 한 파일 저장\')\n    bundle = output / \'CosyVoice3_audio.zip\'\n    with zipfile.ZipFile(bundle, \'w\', compression=zipfile.ZIP_DEFLATED) as archive:\n        archive.write(mp3, \'full_audio.mp3\')\n        archive.writestr(\'VOICE_ATTRIBUTION.txt\', ATTRIBUTION)\n    print(\'✅ 완성: \' + str(bundle), flush=True)\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'command\', choices=[\'setup\', \'start\', \'serve\', \'tunnel\', \'direct\'])\n    parser.add_argument(\'--plan\')\n    args = parser.parse_args()\n    if args.command == \'direct\':\n        direct(args.plan)\n    else:\n        globals()[args.command]()\n', encoding='utf-8')
(ROOT / 'cosy3_model.py').write_text('"""CosyVoice 3 reference bank and official FP32 inference, used only in Colab."""\nfrom collections import OrderedDict\nimport hashlib\nimport math\nfrom pathlib import Path\nimport re\nimport threading\nimport unicodedata\n\nfrom cosy3_voicebank_catalog import VOICEBANK, VOICE_REPO, VOICE_REVISION\n\nPREFIX = \'You are a helpful assistant.\'\nEND = \'<|endofprompt|>\'\n\n\ndef spoken(value, limit=12000):\n    value = unicodedata.normalize(\'NFC\', str(value)).strip()\n    value = re.sub(\'[\\u200b\\u200c\\u200d\\ufeff]\', \'\', value)\n    if not value or len(value) > limit or not any(c.isalnum() for c in value):\n        raise ValueError(\'대사는 비어 있지 않은 12,000자 이하의 문장이어야 합니다.\')\n    if \'<|\' in value or \'|>\' in value:\n        raise ValueError(\'대사에는 모델 제어 기호를 넣을 수 없습니다.\')\n    return value\n\n\ndef split_text(text, tokenizer):\n    """Keep Korean text intact; bound tokens without the English normalizer."""\n    def count(value):\n        return len(tokenizer.encode(value, allowed_special=\'all\'))\n    result, current = [], \'\'\n    for word in text.split():\n        if current and count(current + \' \' + word) > 90:\n            result.append(current)\n            current = \'\'\n        while count(word) > 90:\n            low, high = 1, len(word)\n            while low < high:\n                middle = (low + high + 1) // 2\n                if count(word[:middle]) <= 90:\n                    low = middle\n                else:\n                    high = middle - 1\n            result.append(word[:low])\n            word = word[low:]\n        current = (current + \' \' + word).strip()\n    if current:\n        result.append(current)\n    if len(result) > 1 and count(result[-1]) < 20 and count(result[-2] + \' \' + result[-1]) <= 110:\n        result[-2:] = [result[-2] + \' \' + result[-1]]\n    return result\n\n\nclass Cosy3Model:\n    def __init__(self, model_dir, root):\n        import torch\n        from cosyvoice.cli.cosyvoice import CosyVoice3\n        if not torch.cuda.is_available():\n            raise RuntimeError(\'CosyVoice 3는 이 코랩의 GPU가 필요합니다. 런타임 유형을 T4 GPU로 바꿔주세요.\')\n        torch.set_num_threads(1)\n        torch.set_num_interop_threads(1)\n        torch.backends.cuda.matmul.allow_tf32 = False\n        torch.backends.cudnn.allow_tf32 = False\n        self.root = Path(root)\n        self.model = CosyVoice3(model_dir=str(model_dir), load_trt=False, load_vllm=False, fp16=False)\n        self.sample_rate = self.model.sample_rate\n        self.reference_lock = threading.RLock()\n        self.cache = OrderedDict()\n        self.gpu = torch.cuda.get_device_name(0)\n\n    def reference(self, voice):\n        from huggingface_hub import hf_hub_download\n        if voice not in VOICEBANK:\n            raise ValueError(\'CosyVoice 3 목소리를 다시 선택해주세요.\')\n        return hf_hub_download(VOICE_REPO, filename=VOICEBANK[voice][\'filename\'],\n                               revision=VOICE_REVISION, cache_dir=str(self.root / \'voice_cache\'))\n\n    def conditioning(self, path, transcript):\n        import soundfile as sf\n        import torch\n        digest = hashlib.sha256(Path(path).read_bytes() + transcript.encode()).hexdigest()\n        with self.reference_lock, torch.inference_mode():\n            if digest in self.cache:\n                self.cache.move_to_end(digest)\n                return self.cache[digest], True\n            audio, rate = sf.read(path, dtype=\'float32\', always_2d=True)\n            import numpy as np\n            duration = len(audio) / rate\n            if not 3 <= duration <= 30 or not np.isfinite(audio).all() or np.max(np.abs(audio)) < 0.001:\n                raise ValueError(\'참고 음성은 잡음·배경음 없이 말한 3~30초 녹음이어야 합니다.\')\n            # Upstream performs its own resampling. No pitch/rate conversion.\n            prepared = self.model.frontend.frontend_zero_shot(\n                \'\', PREFIX + END + transcript, str(path), self.sample_rate, \'\')\n            prepared.pop(\'text\', None)\n            prepared.pop(\'text_len\', None)\n            prepared = {key: value.detach().cpu() if torch.is_tensor(value) else value\n                        for key, value in prepared.items()}\n            self.cache[digest] = prepared\n            while len(self.cache) > 64:\n                self.cache.popitem(last=False)\n            return prepared, False\n\n    def generate(self, text, voice, *, style=\'\', speed=1.0, ref_path=\'\', prompt_text=\'\', progress=None):\n        import numpy as np\n        import torch\n        text = spoken(text)\n        if not math.isfinite(speed) or not 0.8 <= speed <= 1.2:\n            raise ValueError(\'읽기 속도는 0.8~1.2 범위에서 선택해주세요.\')\n        style = str(style).strip()\n        if len(style) > 1200 or \'<|\' in style or \'|>\' in style:\n            raise ValueError(\'음성 스타일 지시가 올바르지 않습니다.\')\n        if voice == \'custom\':\n            prompt_text = spoken(prompt_text, 2000)\n            if not ref_path:\n                raise ValueError(\'한국어 참고 음성을 등록해주세요.\')\n            reference = ref_path\n        else:\n            reference = self.reference(voice)\n            prompt_text = \'\'  # cross-language: never pair an invented transcript.\n        prepared, cached = self.conditioning(reference, prompt_text)\n        parts = split_text(text, self.model.frontend.tokenizer)\n        outputs = []\n        with torch.inference_mode():\n            for number, part in enumerate(parts, 1):\n                if progress:\n                    progress(number - 1, len(parts), cached)\n                inputs = {key: value.to(self.model.frontend.device) if torch.is_tensor(value) else value\n                          for key, value in prepared.items()}\n                if style:\n                    instruction = PREFIX + \' Speak Korean. Preserve the reference speaker identity. \' + style + END\n                    inputs[\'prompt_text\'], inputs[\'prompt_text_len\'] = self.model.frontend._extract_text_token(instruction)\n                    inputs.pop(\'llm_prompt_speech_token\', None)\n                    inputs.pop(\'llm_prompt_speech_token_len\', None)\n                    spoken_part = part\n                elif voice != \'custom\':\n                    # Official CosyVoice 3 cross-language prefix belongs to text.\n                    for key in (\'prompt_text\', \'prompt_text_len\', \'llm_prompt_speech_token\', \'llm_prompt_speech_token_len\'):\n                        inputs.pop(key, None)\n                    spoken_part = PREFIX + END + part\n                else:\n                    spoken_part = part\n                inputs[\'text\'], inputs[\'text_len\'] = self.model.frontend._extract_text_token(spoken_part)\n                # Model weights, FP32, flow steps and sampling remain upstream defaults.\n                chunks = [row[\'tts_speech\'].detach().float().cpu().reshape(-1).numpy()\n                          for row in self.model.model.tts(**inputs, stream=False, speed=speed)]\n                if not chunks:\n                    raise RuntimeError(\'모델이 음성을 반환하지 않았습니다.\')\n                samples = np.concatenate(chunks)\n                if not len(samples) or not np.isfinite(samples).all() or np.max(np.abs(samples)) < 0.00001:\n                    raise RuntimeError(\'정상적인 음성이 생성되지 않았습니다. 자동 재생성하지 않았습니다.\')\n                outputs.append(samples)\n                if progress:\n                    progress(number, len(parts), cached)\n        return np.concatenate(outputs), self.sample_rate\n', encoding='utf-8')
(ROOT / 'cosy3_voicebank_catalog.py').write_text('"""Stable reference identities for CosyVoice 3; no synthesis on import.\n\nThese are distinct VCTK speakers, not pitch-shifted copies or Gemini voices.\nThe English recordings condition Korean cross-language synthesis. UI labels\ndeliberately make no unverified claims about age or Korean pronunciation.\n"""\nMODEL_ID = \'FunAudioLLM/Fun-CosyVoice3-0.5B-2512\'\nMODEL_REVISION = \'29e01c4e8d000f4bcd70751be16fa94bf3d85a18\'\nSOURCE_REVISION = \'074ca6dc9e80a2f424f1f74b48bdd7d3fea531cc\'\nSERVICE = \'voice-studio-cosyvoice3\'\nPROTOCOL = \'cosy3-jobs-v1\'\nBANK_REVISION = \'vctk20-20261001-v1\'\nVOICE_REPO = \'kyutai/tts-voices\'\nVOICE_REVISION = \'a0de156151266cf8eb27ac8f27312f7aff2ef7b8\'\nATTRIBUTION = (\n    \'Reference recordings: CSTR VCTK Corpus, Christophe Veaux, Junichi Yamagishi \'\n    \'and Kirsten MacDonald, University of Edinburgh. CC BY 4.0. \'\n    \'https://doi.org/10.7488/ds/2645\\n\'\n    \'Reference distribution: Kyutai tts-voices, vctk/*_023.wav (mic1). \'\n    \'https://huggingface.co/kyutai/tts-voices\\n\'\n    \'License: https://creativecommons.org/licenses/by/4.0/\\n\'\n    \'Output: new Korean synthetic speech conditioned on the reference; \'\n    \'not an original recording or an endorsement by its speaker.\\n\'\n)\n\n_SPEAKERS = {\n    \'남성\': (\'p226\', \'p227\', \'p232\', \'p237\', \'p241\', \'p243\', \'p245\', \'p246\', \'p247\', \'p251\'),\n    \'여성\': (\'p225\', \'p228\', \'p229\', \'p230\', \'p233\', \'p234\', \'p236\', \'p238\', \'p239\', \'p240\'),\n}\nVOICEBANK = {\n    f\'{"M" if gender == "남성" else "F"}{number:02d}\': {\n        \'name\': f\'{gender} {number:02d} · {speaker}\',\n        \'gender\': gender, \'source_speaker\': speaker,\n        \'filename\': f\'vctk/{speaker}_023.wav\', \'reference_language\': \'en\',\n        \'description\': \'서로 다른 실제 화자의 공개 녹음 · 영어 참고 음색으로 한국어 합성\',\n    }\n    for gender, speakers in _SPEAKERS.items()\n    for number, speaker in enumerate(speakers, 1)\n}\n\n\ndef reference_url(voice):\n    return f\'https://huggingface.co/{VOICE_REPO}/resolve/{VOICE_REVISION}/{VOICEBANK[voice]["filename"]}\'\n', encoding='utf-8')
(ROOT / 'colab_session.py').write_text('"""Supervise an actual TTS server and its tunnel from the running notebook cell.\n\nNo synthetic activity, browser clicks, inference or model restarts. This cannot\noverride Colab\'s idle/resource policies. Importing the module does no work.\n"""\nfrom datetime import datetime, timezone\nimport fcntl\nimport json\nfrom pathlib import Path\nimport re\nimport shutil\nimport subprocess\nimport time\nimport urllib.error\nimport urllib.request\n\nVERSION = \'1.0.0\'\nENGINES = {\n    \'qwen-bank\': (\'/content/voice_studio_qwen_bank_v1\', \'voice-studio-qwen-voicebank\', \'Qwen 20종\'),\n    \'cosyvoice\': (\'/content/ai_voice_dual_v1/cosyvoice\', \'ai-voice-studio-cosyvoice\', \'CosyVoice\'),\n    \'cosyvoice3\': (\'/content/voice_studio_cosy3_v1\', \'voice-studio-cosyvoice3\', \'CosyVoice 3\'),\n    \'gpt-sovits\': (\'/content/ai_voice_sovits_v4_1\', \'ai-voice-studio-gpt-sovits\', \'GPT-SoVITS\'),\n}\n\n\ndef _redact(value):\n    return re.sub(r\'/v1/[A-Za-z0-9_-]+\', \'/v1/[접속 코드 숨김]\', str(value))\n\n\ndef _state(root):\n    try:\n        result = json.loads((root / \'state.json\').read_text())\n        if not isinstance(result, dict):\n            raise ValueError()\n        port = result[\'port\']\n        if type(port) is not int or not 1 <= port <= 65535:\n            raise ValueError()\n        base = result.get(\'base\') or f"http://127.0.0.1:{port}/v1/{result[\'token\']}"\n        if not re.fullmatch(rf\'http://127\\.0\\.0\\.1:{port}/v1/[A-Za-z0-9_-]{{20,100}}\', base):\n            raise ValueError()\n        return result, base\n    except (OSError, ValueError, KeyError, TypeError):\n        raise RuntimeError(\'서버 실행 기록을 읽지 못했습니다. 같은 코랩에서 서버 준비 셀을 먼저 실행해주세요.\') from None\n\n\ndef _public(state):\n    value = state.get(\'public_url\') or state.get(\'public_base\') or \'\'\n    return value if re.fullmatch(r\'https://[a-z0-9-]+\\.trycloudflare\\.com/v1/[A-Za-z0-9_-]{20,100}\', value) else \'\'\n\n\ndef _process(pid):\n    """A zombie is stopped; start ticks distinguish a reused process id."""\n    if type(pid) is not int or pid <= 0:\n        return None\n    try:\n        fields = Path(f\'/proc/{pid}/stat\').read_text().rsplit(\') \', 1)[1].split()\n        return None if fields[0] in (\'Z\', \'X\') else (pid, fields[19])\n    except (OSError, IndexError):\n        return None\n\n\nclass _NoRedirect(urllib.request.HTTPRedirectHandler):\n    def redirect_request(self, req, fp, code, msg, headers, newurl):\n        return None\n\n\ndef _health(base, service):\n    try:\n        with urllib.request.build_opener(_NoRedirect()).open(base + \'/health\', timeout=8) as response:\n            data = json.loads(response.read(65537))\n        if not isinstance(data, dict) or data.get(\'service\') != service or data.get(\'ready\') is not True:\n            return False, \'서버 종류 또는 준비 상태가 다릅니다.\'\n        return True, \'정상\'\n    except urllib.error.HTTPError as exc:\n        return False, f\'HTTP {exc.code}\'\n    except (OSError, ValueError):\n        return False, \'응답 없음\'\n\n\ndef _resources(root):\n    info = {}\n    try:\n        available = re.search(r\'^MemAvailable:\\s+(\\d+)\', Path(\'/proc/meminfo\').read_text(), re.M)\n        if available:\n            info[\'available_ram_mib\'] = int(available[1]) // 1024\n        info[\'free_disk_mib\'] = shutil.disk_usage(root).free // (1024 * 1024)\n        # Only report an observed cgroup OOM count, never infer Colab\'s reason.\n        for line in Path(\'/sys/fs/cgroup/memory.events\').read_text().splitlines():\n            key, value = line.split()\n            if key == \'oom_kill\':\n                info[\'oom_kill_count\'] = int(value)\n    except (OSError, ValueError):\n        pass\n    return info\n\n\ndef _record(root, kind, message, **details):\n    path = root / \'session.log\'\n    try:\n        if path.exists() and path.stat().st_size > 1024 * 1024:\n            path.replace(root / \'session.previous.log\')\n        with path.open(\'a\', encoding=\'utf-8\') as log:\n            log.write(json.dumps(dict(time=datetime.now(timezone.utc).isoformat(), kind=kind,\n                                      message=_redact(message), **details), ensure_ascii=False) + \'\\n\')\n        path.chmod(0o600)\n    except OSError:\n        pass  # A full disk must not stop an otherwise working server session.\n\n\ndef _last_errors(root):\n    for filename in (\'server.log\', \'tunnel.log\'):\n        path = root / filename\n        try:\n            with path.open(\'rb\') as log:\n                log.seek(max(0, path.stat().st_size - 6000))\n                output = log.read().decode(\'utf-8\', errors=\'replace\')\n            print(f\'\\n【{filename} 마지막 기록】\\n\' + _redact(\'\\n\'.join(output.splitlines()[-25:])), flush=True)\n        except OSError:\n            pass\n\n\ndef _restore_tunnel(root, state, base, service):\n    """Replace a confirmed stopped tunnel only; never stop the voice model."""\n    binary = root / \'cloudflared\'\n    if not binary.is_file():\n        raise RuntimeError(\'연결 도구가 없습니다. 이 코랩의 4번 셀을 다시 실행해주세요.\')\n    if _process(state.get(\'tunnel_pid\')) is not None:\n        raise RuntimeError(\'기존 연결 도구가 실행 중이므로 중복 실행하지 않았습니다.\')\n    with (root / \'tunnel.log\').open(\'w\') as log:\n        child = subprocess.Popen([str(binary), \'tunnel\', \'--no-autoupdate\', \'--url\',\n                                  f"http://127.0.0.1:{state[\'port\']}"], stdin=subprocess.DEVNULL,\n                                 stdout=log, stderr=subprocess.STDOUT, start_new_session=True)\n    saved = False\n    try:\n        deadline = time.monotonic() + 120\n        while time.monotonic() < deadline:\n            if child.poll() is not None:\n                raise RuntimeError(\'외부 연결 도구가 다시 종료됐습니다. 아래 tunnel.log를 확인해주세요.\')\n            content = (root / \'tunnel.log\').read_text(errors=\'replace\')[-24000:]\n            match = re.search(r\'https://[a-z0-9-]+\\.trycloudflare\\.com\', content)\n            if match:\n                public = match[0] + base.split(f":{state[\'port\']}", 1)[1]\n                ok, _ = _health(public, service)\n                if ok:\n                    current, current_base = _state(root)\n                    if current_base != base:\n                        raise RuntimeError(\'서버 실행 정보가 바뀌었습니다. 중복 연결 생성을 중단했습니다.\')\n                    current.update(tunnel_pid=child.pid)\n                    current[\'public_url\' if \'token\' in current else \'public_base\'] = public\n                    temporary = root / \'state.session.tmp\'\n                    temporary.write_text(json.dumps(current), encoding=\'utf-8\')\n                    temporary.chmod(0o600)\n                    temporary.replace(root / \'state.json\')\n                    saved = True\n                    return current\n            time.sleep(2)\n        raise RuntimeError(\'외부 연결 복구 응답이 없습니다. 음성 서버는 중지하지 않았습니다.\')\n    finally:\n        if not saved and child.poll() is None:\n            child.terminate()\n\n\ndef supervise(engine):\n    """Keep real process supervision attached to the interactive notebook cell."""\n    if engine not in ENGINES:\n        raise ValueError(\'지원하지 않는 음성 엔진입니다.\')\n    directory, service, label = ENGINES[engine]\n    root = Path(directory)\n    state, base = _state(root)\n    with (root / \'session.lock\').open(\'a\') as session_lock:\n        try:\n            fcntl.flock(session_lock, fcntl.LOCK_EX | fcntl.LOCK_NB)\n        except BlockingIOError:\n            print(\'이미 다른 셀에서 서버 상태를 확인하고 있습니다. 해당 셀을 계속 사용하세요.\', flush=True)\n            return\n        ok, reason = _health(base, service)\n        if not ok:\n            _last_errors(root)\n            raise RuntimeError(\'음성 서버 응답 확인 실패: \' + reason + \' 서버 준비 셀의 오류를 확인해주세요.\')\n        server_process = _process(state.get(\'pid\'))\n        initial_resources = _resources(root)\n        _record(root, \'start\', label + \' 서버 감독 시작\', resources=initial_resources)\n        print(f\'\\n▶ {label} 서버 감독 v{VERSION} 실행 중 — 이 셀을 실행 상태로 두고 사이트를 사용하세요.\', flush=True)\n        print(\'서버·외부 연결의 실제 상태와 오류를 확인합니다. 다른 셀을 사용하려면 먼저 이 셀의 ■를 누르세요.\', flush=True)\n        print(\'■는 감독만 종료합니다. 음성 서버·생성 중인 작업·완료 파일은 그대로 둡니다.\', flush=True)\n        print(\'코랩의 자원 회수나 유휴 정책을 변경하지 않으며 최대 사용 시간을 보장하지 않습니다.\', flush=True)\n        repairs, failures, previous, last_report, next_public = 0, 0, \'\', 0.0, 0.0\n        public_reason = \'확인 중\'\n        try:\n            while True:\n                current, current_base = _state(root)\n                if current_base != base:\n                    raise RuntimeError(\'음성 서버 주소가 변경됐습니다. 새 주소의 4번 셀에서 다시 연결해주세요.\')\n                current_process = _process(current.get(\'pid\'))\n                if server_process and current_process != server_process:\n                    resources = _resources(root)\n                    oom_before = initial_resources.get(\'oom_kill_count\', 0)\n                    if resources.get(\'oom_kill_count\', 0) > oom_before:\n                        reason = \'서버 프로세스 종료와 시스템 RAM 부족 종료 기록의 증가가 확인됐습니다.\'\n                    else:\n                        reason = \'음성 서버 프로세스가 종료되거나 교체됐습니다. 종료 원인은 마지막 로그를 확인해야 합니다.\'\n                    _record(root, \'server_stopped\', reason, resources=resources)\n                    raise RuntimeError(reason + \' 코랩 런타임 전체 종료와는 다릅니다.\')\n                ok, reason = _health(base, service)\n                failures = 0 if ok else failures + 1\n                if failures >= 3:\n                    raise RuntimeError(\'음성 서버가 세 번 연속 응답하지 않습니다. 프로세스가 살아 있어 자동 재시작하지 않았습니다. \' + reason)\n                public = _public(current)\n                tunnel_alive = _process(current.get(\'tunnel_pid\'))\n                if ok and type(current.get(\'tunnel_pid\')) is int and tunnel_alive is None:\n                    if repairs >= 2:\n                        raise RuntimeError(\'외부 연결 도구가 반복 종료되어 복구를 멈췄습니다. 음성 서버는 그대로 실행 중입니다.\')\n                    repairs += 1\n                    print(f\'외부 연결 도구 종료 확인 — 음성 서버를 유지하고 연결만 복구합니다 ({repairs}/2).\', flush=True)\n                    _record(root, \'tunnel_stopped\', \'외부 연결만 복구\', attempt=repairs)\n                    current = _restore_tunnel(root, current, base, service)\n                    public = _public(current)\n                    print(\'\\n✅ 새 프로그램 연결 주소 — 사이트의 내 코랩 주소를 아래 주소로 바꿔주세요.\', flush=True)\n                    print(public + (\'/tts\' if engine == \'gpt-sovits\' else \'\'), flush=True)\n                    next_public = 0\n                now = time.monotonic()\n                if ok and now >= next_public:\n                    public_ok, public_reason = _health(public, service) if public else (False, \'주소 없음\')\n                    public_reason = \'정상\' if public_ok else public_reason + \' · 코랩 내부 서버는 정상\'\n                    next_public = now + 60\n                message = f\'음성 서버: {reason} · 외부 연결: {public_reason}\'\n                if message != previous or now - last_report >= 60:\n                    print(datetime.now().strftime(\'%H:%M:%S\') + \' · \' + message, flush=True)\n                    _record(root, \'status\', message, resources=_resources(root))\n                    previous, last_report = message, now\n                time.sleep(15)\n        except KeyboardInterrupt:\n            _record(root, \'user_stop\', \'사용자가 감독 셀을 중단함\')\n            print(\'\\n감독을 종료했습니다. 음성 서버와 생성 결과는 유지됩니다. 다시 감독하려면 4번 셀을 실행하세요.\', flush=True)\n        except Exception as exc:\n            _record(root, \'error\', str(exc), resources=_resources(root))\n            _last_errors(root)\n            raise\n\n\ndef diagnostics(engine):\n    root = Path(ENGINES[engine][0])\n    print(\'현재 자원 상태:\', _resources(root))\n    path = root / \'session.log\'\n    if path.is_file():\n        with path.open(\'rb\') as log:\n            log.seek(max(0, path.stat().st_size - 12000))\n            print(_redact(log.read().decode(\'utf-8\', errors=\'replace\')))\n    else:\n        print(\'이 런타임에는 감독 기록이 없습니다. 런타임 자체가 삭제됐다면 이전 로컬 기록은 복구할 수 없습니다.\')\n    _last_errors(root)\n', encoding='utf-8')
def run_visible(command):
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
        code = process.wait()
    except BaseException:
        process.terminate()
        process.wait()
        raise
    if code:
        raise RuntimeError('위에 표시된 실제 오류를 확인해주세요. 실행이 중단되었습니다. 종료 코드: ' + str(code))

run_visible([sys.executable, '-u', str(ROOT / 'cosy3_colab_server.py'), 'setup'])


In [ ]:
#@title 2. CosyVoice 3 모델·서버 준비 (음성 생성 없음)
from pathlib import Path
import subprocess, sys
ROOT = Path('/content/voice_studio_cosy3_v1')
def run_visible(command):
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
        code = process.wait()
    except BaseException:
        process.terminate()
        process.wait()
        raise
    if code:
        raise RuntimeError('위에 표시된 실제 오류를 확인해주세요. 실행이 중단되었습니다. 종료 코드: ' + str(code))

run_visible([sys.executable, '-u', str(ROOT / 'cosy3_colab_server.py'), 'start'])


In [ ]:
#@title 3. 코랩에서 직접 전체 생성 · 사이트에서 받은 CosyVoice3_대본.zip 업로드
from pathlib import Path
from google.colab import files
import io, subprocess, sys, zipfile
ROOT = Path('/content/voice_studio_cosy3_v1')
print('사이트에서 전체 화자를 CosyVoice 3로 설정한 뒤 받은 대본 ZIP을 올리세요.')
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError('대본 ZIP 파일 하나만 올려주세요.')
name, data = next(iter(uploaded.items()))
with zipfile.ZipFile(io.BytesIO(data)) as archive:
    entry = archive.getinfo('plan.json')
    if entry.file_size > 64 * 1024 * 1024:
        raise ValueError('대본 파일이 너무 큽니다.')
    plan = archive.read(entry)
plan_path = ROOT / 'direct_plan.json'
plan_path.write_bytes(plan)
plan_path.chmod(0o600)
def run_visible(command):
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
        code = process.wait()
    except BaseException:
        process.terminate()
        process.wait()
        raise
    if code:
        raise RuntimeError('위에 표시된 실제 오류를 확인해주세요. 실행이 중단되었습니다. 종료 코드: ' + str(code))

run_visible([sys.executable, '-u', str(ROOT / 'cosy3_colab_server.py'), 'direct', '--plan', str(plan_path)])
files.download(str(ROOT / 'direct_results/CosyVoice3_audio.zip'))


In [ ]:
#@title 4. 사이트 연결 주소 표시 · 주소를 복사한 뒤 실행 상태로 유지 (v1.0.1)
# CosyVoice 3 주소 다시 표시 · 기존 4번의 ■를 누르고 새 코드 셀에서 실행
# 현재 모델과 생성 파일을 유지합니다. 설치·음성 생성을 실행하지 않습니다.
from pathlib import Path
from runpy import run_path
import sys

ROOT = Path('/content/voice_studio_cosy3_v1')
if not (ROOT / 'cosy3_colab_server.py').is_file():
    raise RuntimeError('CosyVoice 3의 1번·2번을 실행한 코랩에서 사용해주세요.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Run in the notebook process so the connection address is actually visible.
server = run_path(str(ROOT / 'cosy3_colab_server.py'))
server['tunnel']()
print('\n위의 https:// 주소 전체를 사이트의 ‘CosyVoice 3 · 내 코랩 주소’에 넣으세요.', flush=True)
print('주소가 나온 뒤에는 이 셀이 계속 실행되는 것이 정상입니다.', flush=True)
run_path(str(ROOT / 'colab_session.py'))['supervise']('cosyvoice3')


In [ ]:
#@title 5. 오류 로그 확인 (먼저 4번 셀의 ■를 누르세요)
from pathlib import Path
from runpy import run_path
ROOT = Path('/content/voice_studio_cosy3_v1')
run_path(str(ROOT / 'colab_session.py'))['diagnostics']('cosyvoice3')
for name in ('setup.log', 'server.log', 'tunnel.log'):
    path = ROOT / name
    print('【' + name + '】')
    if path.is_file():
        with path.open('rb') as stream:
            stream.seek(max(0, path.stat().st_size - 8000))
            print(stream.read().decode(errors='replace'))
    else:
        print('아직 기록 없음')


### 목소리 출처와 생성 방식

- 참고 음성: CSTR VCTK Corpus, Christophe Veaux, Junichi Yamagishi, Kirsten MacDonald (University of Edinburgh), **CC BY 4.0**. [원본](https://doi.org/10.7488/ds/2645)
- 배포: [Kyutai tts-voices의 VCTK](https://huggingface.co/kyutai/tts-voices). 다른 데이터셋의 비상업용 음성은 포함하지 않습니다.
- 기본 음색은 실제 서로 다른 20명의 녹음을 고정해 사용합니다. Gemini 성우나 가상의 한국인 성우 이름으로 표시하지 않습니다.
- 공식 [CosyVoice 3](https://github.com/QwenAudio/CosyVoice)의 교차 언어 합성 / 감정 지시 / 한국어 참고 음성 복제를 사용합니다.
- 버전 3 전용 GPU 계산은 한 대사씩 처리하고, 완료 파일의 전송과 다음 계산을 겹칩니다. 사이트에서는 버전 2·Gemini와 별도 대기열로 실행합니다.
- 전체 결과는 대본 번호순으로 MP3 하나로 합칩니다. 출처 문구가 ZIP에 들어 있습니다.
- 이 배포는 요청에 따라 테스트·모델 실행·합성 없이 소스를 작성했습니다. 실제 GPU 음질·속도는 확인하지 않았습니다.
